In [4]:
import re

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import spacy
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
# Load data
bbc_data = pd.read_csv("Files/bbc_news.csv")
bbc_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   1000 non-null   int64 
 1   index        1000 non-null   int64 
 2   title        1000 non-null   object
 3   pubDate      1000 non-null   object
 4   guid         1000 non-null   object
 5   link         1000 non-null   object
 6   description  1000 non-null   object
dtypes: int64(2), object(5)
memory usage: 54.8+ KB


In [6]:
titles = pd.DataFrame(bbc_data["title"]) # Series
titles.head()

,title
0,Can I refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...
2,Rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...


### Clean Data

In [7]:
# Lowercase
titles["lowercase"] = titles['title'].str.lower()
titles.head()

,title,lowercase
0,Can I refuse to work?,can i refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...


In [8]:
# Remove stop words
en_stopwords = stopwords.words("english")
titles["no_stopwords"] = titles.lowercase.apply(lambda x: " ".join(word for word in x.split() if word not in en_stopwords))
titles["no_stopwords"]

0                                           refuse work?
1      'liz truss brief?' world reacts uk political t...
2        rationing energy nothing new off-grid community
3          hunt superyachts sanctioned russian oligarchs
4            platinum jubilee: 70 years queen 70 seconds
                             ...                        
995    dominic raab: third senior civil servant gives...
996                  highlights: radacanu beats uytvanck
997         pictures: mountain bikers descend snowy peak
998    companies must help cut living costs, says new...
999       beware online car sale scams, consumers warned
Name: no_stopwords, Length: 1000, dtype: object

In [9]:
# Remove punctuations
titles["no_punc"] = titles.apply(lambda x: re.sub(r"[^\w\s]", "", x.no_stopwords), axis=1)
titles.no_punc

0                                            refuse work
1      liz truss brief world reacts uk political turmoil
2         rationing energy nothing new offgrid community
3          hunt superyachts sanctioned russian oligarchs
4             platinum jubilee 70 years queen 70 seconds
                             ...                        
995    dominic raab third senior civil servant gives ...
996                   highlights radacanu beats uytvanck
997          pictures mountain bikers descend snowy peak
998    companies must help cut living costs says new ...
999        beware online car sale scams consumers warned
Name: no_punc, Length: 1000, dtype: object

In [10]:
# Tokenize
titles["raw_tokens"] = titles["title"].apply(lambda x: word_tokenize(x))
titles["clean_tokens"] = titles["no_punc"].apply(lambda x: word_tokenize(x))
titles[["raw_tokens", "clean_tokens"]]

,raw_tokens,clean_tokens
0,"[Can, I, refuse, to, work, ?]","[refuse, work]"
1,"['Liz, Truss, the, Brief, ?, ', World, reacts,...","[liz, truss, brief, world, reacts, uk, politic..."
2,"[Rationing, energy, is, nothing, new, for, off...","[rationing, energy, nothing, new, offgrid, com..."
3,"[The, hunt, for, superyachts, of, sanctioned, ...","[hunt, superyachts, sanctioned, russian, oliga..."
4,"[Platinum, Jubilee, :, 70, years, of, the, Que...","[platinum, jubilee, 70, years, queen, 70, seco..."
...,...,...
995,"[Dominic, Raab, :, Third, senior, civil, serva...","[dominic, raab, third, senior, civil, servant,..."
996,"[Highlights, :, Radacanu, beats, Uytvanck]","[highlights, radacanu, beats, uytvanck]"
997,"[In, pictures, :, Mountain, bikers, descend, s...","[pictures, mountain, bikers, descend, snowy, p..."
998,"[Companies, must, help, cut, living, costs, ,,...","[companies, must, help, cut, living, costs, sa..."


In [11]:
# Lemmatization
lemmatizer = WordNetLemmatizer()
titles["clean_lemmatized"] = titles["clean_tokens"].apply(lambda tokens: [lemmatizer.lemmatize(token) for token in tokens])
titles[["clean_lemmatized", "raw_tokens"]]

,clean_lemmatized,raw_tokens
0,"[refuse, work]","[Can, I, refuse, to, work, ?]"
1,"[liz, truss, brief, world, reacts, uk, politic...","['Liz, Truss, the, Brief, ?, ', World, reacts,..."
2,"[rationing, energy, nothing, new, offgrid, com...","[Rationing, energy, is, nothing, new, for, off..."
3,"[hunt, superyachts, sanctioned, russian, oliga...","[The, hunt, for, superyachts, of, sanctioned, ..."
4,"[platinum, jubilee, 70, year, queen, 70, second]","[Platinum, Jubilee, :, 70, years, of, the, Que..."
...,...,...
995,"[dominic, raab, third, senior, civil, servant,...","[Dominic, Raab, :, Third, senior, civil, serva..."
996,"[highlight, radacanu, beat, uytvanck]","[Highlights, :, Radacanu, beats, Uytvanck]"
997,"[picture, mountain, bikers, descend, snowy, peak]","[In, pictures, :, Mountain, bikers, descend, s..."
998,"[company, must, help, cut, living, cost, say, ...","[Companies, must, help, cut, living, costs, ,,..."


In [12]:
tokens_raw_list = sum(titles.raw_tokens, [])
tokens_clean_list = sum(titles.clean_lemmatized, [])
print(len(tokens_raw_list), len(tokens_clean_list))

11219 7520


### POST (Parts of Speech Tagging)

In [13]:
nlp = spacy.load("en_core_web_sm")
spacy_doc = nlp(" ".join(tokens_raw_list)) # Make them a large sentence and pass them

# Create dataframe for tagging the parts of speech
pos_df = pd.DataFrame(columns=["token", "part_of_speech"])
for token in spacy_doc:
    # This is not memory friendly and takes a longer time to run. Creates a new dataframe each time
    '''
    pos_df = pd.concat([
        pos_df,
        pd.DataFrame([{"token": token.text, "part_of_speech": token.pos_}])
    ])
    '''
    # This is better
    pos_df = pd.concat([
        pos_df,
        pd.DataFrame.from_records([{"token": token.text, "part_of_speech": token.pos_}])
    ], ignore_index=True)

pos_df

,token,part_of_speech
0,Can,AUX
1,I,PRON
2,refuse,VERB
3,to,PART
4,work,VERB
...,...,...
11742,sale,NOUN
11743,scams,NOUN
11744,",",PUNCT
11745,consumers,NOUN


In [14]:
pos_df_counts = pos_df.groupby(["token", "part_of_speech"]).size().reset_index(name="counts").sort_values(by="counts", ascending=False)
pos_df_counts

,token,part_of_speech,counts
95,:,PUNCT,543
8,',PUNCT,300
2897,in,ADP,187
4082,to,PART,175
3268,of,ADP,172
...,...,...,...
2304,crumbling,VERB,1
2305,crunch,PROPN,1
827,Jarrod,PROPN,1
826,Japanese,ADJ,1


In [15]:
# Let's find the most common nouns
nouns = pos_df_counts[pos_df_counts.part_of_speech == "NOUN"]
nouns  # War's the most common

,token,part_of_speech,counts
4267,war,NOUN,35
3552,record,NOUN,15
3416,police,NOUN,14
4356,year,NOUN,14
4316,win,NOUN,14
...,...,...,...
2294,criticism,NOUN,1
2296,crocodile,NOUN,1
2297,crop,NOUN,1
2300,crown,NOUN,1


In [16]:
# POS tagging for the cleaned data
spacy_clean_doc = nlp(" ".join(tokens_clean_list))

pos_clean_df = pd.DataFrame(columns=["token", "pos_tag"])
for token in spacy_clean_doc:
    pos_clean_df = pd.concat([
        pos_clean_df,
        pd.DataFrame.from_records([{
            "token": token.text,
            "pos_tag": token.pos_
        }])
    ], ignore_index=True)


In [20]:
pos_clean_df_counts = pos_clean_df.groupby(["token", "pos_tag"]).size().reset_index(name="counts").sort_values(by="counts", ascending=False)
pos_clean_df_counts

,token,pos_tag,counts
30,2022,NUM,47
1162,england,PROPN,45
870,cup,PROPN,39
3056,say,VERB,37
3707,uk,PROPN,37
...,...,...,...
1559,halftime,NOUN,1
1560,hall,NOUN,1
1562,halloween,PROPN,1
1563,halted,VERB,1


In [21]:
clean_nouns = pos_clean_df_counts[pos_clean_df_counts.pos_tag == "NOUN"]
clean_nouns

,token,pos_tag,counts
3840,war,NOUN,34
3948,world,NOUN,30
2136,man,NOUN,22
907,day,NOUN,21
3973,year,NOUN,20
...,...,...,...
1547,gymnastics,NOUN,1
1552,hail,NOUN,1
1554,hair,NOUN,1
1559,halftime,NOUN,1


### NER (Named Entity Recognition)

In [22]:
ner_df = pd.DataFrame(columns=["token", "ner_tag"])
for token in spacy_doc.ents:
    if pd.isna(token.label_) is False:
        ner_df = pd.concat([
            ner_df,
            pd.DataFrame.from_records([{
                "token": token.text,
                "ner_tag": token.label_
            }])
        ], ignore_index=True)

ner_df

,token,ner_tag
0,Liz Truss,PERSON
1,UK,GPE
2,Rationing,PRODUCT
3,superyachts,CARDINAL
4,Russian,NORP
...,...,...
1665,Frenkie de Jong,PERSON
1666,Manchester United,PERSON
1667,Barcelona,GPE
1668,Dominic Raab,PERSON


In [24]:
ner_df_counts = ner_df.groupby(["token", "ner_tag"]).size().reset_index(name="counts").sort_values(by="counts", ascending=False)
ner_df_counts

,token,ner_tag,counts
965,Ukraine,GPE,47
955,UK,GPE,36
329,England,GPE,32
819,Russian,NORP,20
957,US,GPE,19
...,...,...,...
405,Georgia Taylor-Brown,PERSON,1
406,Geraint Thomas,PERSON,1
409,Ghislaine Maxwell,PERSON,1
410,Gianluigi Lentini,PERSON,1
